# UID-Based PyTorch LSTM Sequence Classifier for IEEE-CIS Fraud

This is the PyTorch version of `UID_LSTM_Strict_Time_Validation.ipynb`. It uses the exported pickle files directly:

- `pkl_exported_files/X_train_copy4.pkl`
- `pkl_exported_files/X_test_copy4.pkl`
- `pkl_exported_files/y_train.pkl`

The modeling frame is the same:

- per-UID sequence classifier, not a global time-series forecaster;
- small UID-history windows;
- current transaction features are included in the final timestep;
- `isFraud` is never included in `X`;
- fold-local preprocessing only;
- strict expanding time validation.

Unlike the Keras notebook, this version uses sequence lengths and `pack_padded_sequence`, so padded timesteps are ignored by the PyTorch LSTM.


In [ ]:
from pathlib import Path
from collections import deque
import gc
import os
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

try:
    from sklearn.metrics import roc_auc_score
except Exception:
    roc_auc_score = None

pd.set_option('display.max_columns', 120)
print('PyTorch:', torch.__version__)


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path('/data/ieee-fraud-detection')
START_DATE = pd.Timestamp('2017-11-30')

# Start small. Most UIDs do not have 30 transactions.
WINDOW_SIZE = 5

# If the pickle has V columns, False drops them during feature selection.
INCLUDE_V = False

# Limit features for LSTM memory. Set None to keep all eligible features.
MAX_FEATURES = 180

# Strict expanding month validation.
MIN_TRAIN_MONTHS = 3
MAX_FOLDS = None   # e.g. 1 for a quick single-fold run

EPOCHS = 8
BATCH_SIZE = 1024
LEARNING_RATE = 1e-3
LSTM_UNITS = 64
DROPOUT = 0.25
PATIENCE = 2

RUN_TEST_PREDICTION = False
OUTPUT_DIR = DATA_DIR / 'model_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('DEVICE:', DEVICE)


## Load exported pickle files

This notebook now starts from your exported objects:

- `pkl_exported_files/X_train_copy4.pkl`
- `pkl_exported_files/X_test_copy4.pkl`
- `pkl_exported_files/y_train.pkl`

So it no longer reloads or reconstructs the full CSV pipeline.


In [ ]:
PKL_DIR = DATA_DIR / 'pkl_exported_files'

def load_exported_pickles(pkl_dir=PKL_DIR):
    train = pd.read_pickle(pkl_dir / 'X_train_copy4.pkl')
    test = pd.read_pickle(pkl_dir / 'X_test_copy4.pkl')
    y = pd.read_pickle(pkl_dir / 'y_train.pkl')

    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    y = pd.Series(y).astype('int8')

    # Normalize row keys. Most exported notebook objects already use TransactionID as index.
    for name, df in [('train', train), ('test', test)]:
        if 'TransactionID' in df.columns and df.index.name != 'TransactionID':
            df.set_index('TransactionID', drop=True, inplace=True)
        print(name, df.shape, 'index name:', df.index.name)

    if 'isFraud' in train.columns:
        train = train.drop(columns=['isFraud'])

    if len(y) != len(train):
        raise ValueError(f'y length {len(y)} does not match train rows {len(train)}')
    if not y.index.equals(train.index):
        # Common after pickling a Series without preserving TransactionID index.
        y.index = train.index

    required = ['TransactionDT', 'D1', 'card1_addr1']
    missing_train = [c for c in required if c not in train.columns]
    missing_test = [c for c in required if c not in test.columns]
    if missing_train or missing_test:
        raise ValueError(f'Missing required columns. train={missing_train}, test={missing_test}')

    return train, test, y

X_train_raw, X_test_raw, y_train = load_exported_pickles()
print('y:', y_train.shape, y_train.mean())
display(X_train_raw.head())


## Ensure `DT`, temporary UID, and time-gap features

The pickle files already contain the feature-engineered `X_train_copy4` / `X_test_copy4` objects. This cell only verifies or recreates the columns needed for UID sequence modeling.

To avoid train/test UID-code mismatch, the notebook rebuilds a fresh `uid_raw_lstm` from your formula and then factorizes train+test together. This factorization uses no labels.


In [ ]:
def add_dt_uid_and_gap_features(train, test):
    train = train.copy()
    test = test.copy()

    for df in [train, test]:
        if 'DT' not in df.columns:
            df['DT'] = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
        else:
            df['DT'] = pd.to_datetime(df['DT'])

        if 'DT_M' not in df.columns:
            df['DT_M'] = ((df['DT'].dt.year - START_DATE.year) * 12 + df['DT'].dt.month).astype('int16')
        if 'DT_D' not in df.columns:
            df['DT_D'] = np.floor(df['TransactionDT'] / (24 * 60 * 60)).astype('int16')
        if 'DT_hour' not in df.columns:
            df['DT_hour'] = df['DT'].dt.hour.astype('int8')
        if 'DT_day_week' not in df.columns:
            df['DT_day_week'] = df['DT'].dt.dayofweek.astype('int8')
        if 'DT_day_month' not in df.columns:
            df['DT_day_month'] = df['DT'].dt.day.astype('int8')
        if 'DT_week_month' not in df.columns:
            df['DT_week_month'] = (((df['DT'].dt.day - 1) // 7) + 1).astype('int8')
        if 'is_december' not in df.columns:
            df['is_december'] = (df['DT'].dt.month == 12).astype('int8')

        df['day'] = np.floor(df.TransactionDT / (24 * 60 * 60))
        df['uid_raw_lstm'] = (
            df.card1_addr1.astype(str)
            + '_'
            + np.floor(df.day - df['D1'].fillna(-999)).astype(str)
        )

        if 'cents' not in df.columns:
            df['cents'] = (df['TransactionAmt'] - np.floor(df['TransactionAmt'])).astype('float32')
        if 'dollars' not in df.columns:
            df['dollars'] = np.floor(df['TransactionAmt']).astype('float32')

    both_uid = pd.concat([train['uid_raw_lstm'], test['uid_raw_lstm']], axis=0)
    uid_codes, uid_uniques = pd.factorize(both_uid, sort=True)
    train['uid_code'] = uid_codes[:len(train)].astype('int32')
    test['uid_code'] = uid_codes[len(train):].astype('int32')

    combined = pd.concat([
        train.assign(_split='train'),
        test.assign(_split='test')
    ], axis=0).sort_values(['uid_code', 'TransactionDT'], kind='mergesort')

    combined['uid_tx_count_so_far'] = combined.groupby('uid_code').cumcount().astype('int16')
    combined['uid_delta_seconds'] = combined.groupby('uid_code')['TransactionDT'].diff().fillna(-1).astype('float32')
    combined['uid_delta_days'] = (combined['uid_delta_seconds'] / 86400).astype('float32')
    combined['uid_prev_amt'] = combined.groupby('uid_code')['TransactionAmt'].shift(1).fillna(-1).astype('float32')
    combined['uid_amt_delta_prev'] = (combined['TransactionAmt'] - combined['uid_prev_amt']).astype('float32')

    train_out = combined[combined['_split'] == 'train'].drop(columns=['_split']).sort_values('DT', kind='mergesort')
    test_out = combined[combined['_split'] == 'test'].drop(columns=['_split']).sort_values('DT', kind='mergesort')
    return train_out, test_out

X_train_base, X_test_base = add_dt_uid_and_gap_features(X_train_raw, X_test_raw)
y_train = y_train.loc[X_train_base.index]

print('Train day range:', X_train_base['DT_D'].min(), X_train_base['DT_D'].max())
print('Test day range:', X_test_base['DT_D'].min(), X_test_base['DT_D'].max())
print('Train DT monotonic:', X_train_base['DT'].is_monotonic_increasing)
print('Test DT monotonic:', X_test_base['DT'].is_monotonic_increasing)
print('UID count train/test:', X_train_base['uid_code'].nunique(), X_test_base['uid_code'].nunique())
display(X_train_base[['DT', 'uid_raw_lstm', 'uid_code', 'uid_tx_count_so_far', 'uid_delta_days', 'TransactionAmt']].head())


## Strict expanding time folds

The first validation fold trains on the first `MIN_TRAIN_MONTHS` months and validates on the next month. Later folds expand the training period.

This differs from `GroupKFold(DT_M)`, which may train on future months to validate an earlier month.

In [ ]:
def expanding_month_folds(df, period_col='DT_M', min_train_months=MIN_TRAIN_MONTHS, max_folds=MAX_FOLDS):
    months = sorted(df[period_col].dropna().astype(int).unique().tolist())
    valid_months = months[min_train_months:]
    if max_folds is not None:
        valid_months = valid_months[-max_folds:]
    folds = []
    for valid_month in valid_months:
        train_months = [m for m in months if m < valid_month]
        idx_train = df.index[df[period_col].isin(train_months)]
        idx_valid = df.index[df[period_col] == valid_month]
        folds.append((train_months, valid_month, idx_train, idx_valid))
    return folds

folds = expanding_month_folds(X_train_base)
for train_months, valid_month, idx_train, idx_valid in folds:
    print(f'valid_month={valid_month}, train_months={train_months}, train_rows={len(idx_train):,}, valid_rows={len(idx_valid):,}')


## Fold-local preprocessing

This cell fits category encoders, missing-value medians, and scaling statistics on the fold's training rows only.

This is intentionally simpler than the XGBoost feature engineering. For LSTM, start with a stable baseline before adding UID aggregates or V columns.

In [ ]:
DROP_COLS = {
    'isFraud', 'DT', 'uid', 'uid_code', 'uid_raw', 'uid_raw_lstm', 'card1_addr1',
}

# Columns that are identifiers or raw strings used only for sequence construction.
ALWAYS_DROP_PREFIXES = []

def candidate_feature_columns(df, include_v=INCLUDE_V):
    cols = []
    for c in df.columns:
        if c in DROP_COLS:
            continue
        if (not include_v) and c.startswith('V'):
            continue
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            continue
        cols.append(c)
    return cols

def fit_preprocessor(train_df, include_v=INCLUDE_V, max_features=MAX_FEATURES):
    feature_cols = candidate_feature_columns(train_df, include_v=include_v)
    cat_cols = [c for c in feature_cols if train_df[c].dtype == 'object' or str(train_df[c].dtype) == 'category']
    num_cols = [c for c in feature_cols if c not in cat_cols]

    encoders = {}
    for c in cat_cols:
        vals = pd.Series(train_df[c].dropna().astype(str).unique())
        encoders[c] = {v: i for i, v in enumerate(vals)}

    tmp = transform_categories(train_df[feature_cols], encoders)
    medians = tmp.median(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0)
    tmp = tmp.replace([np.inf, -np.inf], np.nan).fillna(medians)

    # Optional unsupervised feature limit: keep highest-variance columns using train fold only.
    if max_features is not None and len(feature_cols) > max_features:
        variances = tmp.var(axis=0).sort_values(ascending=False)
        keep = variances.head(max_features).index.tolist()
    else:
        keep = feature_cols

    tmp = tmp[keep]
    means = tmp.mean(axis=0)
    stds = tmp.std(axis=0).replace(0, 1).fillna(1)
    return {
        'feature_cols': keep,
        'encoders': encoders,
        'medians': medians.reindex(keep).fillna(0),
        'means': means,
        'stds': stds,
    }

def transform_categories(df, encoders):
    out = df.copy()
    for c, mp in encoders.items():
        if c in out.columns:
            out[c] = out[c].astype(str).map(mp).fillna(-1).astype('float32')
    for c in out.columns:
        if out[c].dtype == 'object' or str(out[c].dtype) == 'category':
            out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def apply_preprocessor(df, prep):
    cols = prep['feature_cols']
    out = transform_categories(df[cols], prep['encoders'])
    out = out.replace([np.inf, -np.inf], np.nan).fillna(prep['medians'])
    out = ((out - prep['means']) / prep['stds']).astype('float32')

    # Reattach columns needed for sequence grouping/sorting. They are not model features.
    out['uid_code'] = df['uid_code'].values
    out['DT'] = df['DT'].values
    return out


## Build per-UID sequence windows for PyTorch

Each sample is left-aligned and padded at the end:

```text
[real timestep, real timestep, current transaction, padding, padding]
```

The function also returns `lengths`, so PyTorch can use `pack_padded_sequence` and ignore padded timesteps.


In [ ]:
def build_uid_windows(history_df, predict_df, feature_cols, window_size=WINDOW_SIZE):
    """Build left-aligned UID windows and sequence lengths for rows in predict_df."""
    hist = history_df[['uid_code', 'DT'] + feature_cols].copy()
    hist['_predict'] = False
    hist['_txid'] = hist.index

    pred = predict_df[['uid_code', 'DT'] + feature_cols].copy()
    pred['_predict'] = True
    pred['_txid'] = pred.index

    combined = pd.concat([hist, pred], axis=0)
    combined = combined.sort_values(['uid_code', 'DT', '_predict'], kind='mergesort')

    n_pred = len(pred)
    n_feat = len(feature_cols)
    X = np.zeros((n_pred, window_size, n_feat), dtype='float32')
    lengths = np.zeros(n_pred, dtype='int64')
    row_ids = np.empty(n_pred, dtype=pred.index.dtype)

    write_i = 0
    for _, g in combined.groupby('uid_code', sort=False):
        values = g[feature_cols].to_numpy(dtype='float32')
        is_predict = g['_predict'].to_numpy(dtype=bool)
        txids = g['_txid'].to_numpy()
        for pos in np.flatnonzero(is_predict):
            start = max(0, pos - window_size + 1)
            seq = values[start:pos + 1]
            seq_len = len(seq)
            X[write_i, :seq_len, :] = seq
            lengths[write_i] = seq_len
            row_ids[write_i] = txids[pos]
            write_i += 1

    return X[:write_i], lengths[:write_i], row_ids[:write_i]

# Tiny sanity check
toy = pd.DataFrame({
    'uid_code': [1, 1, 1, 2, 1],
    'DT': pd.date_range('2020-01-01', periods=5, freq='h'),
    'f': [10, 11, 12, 20, 13],
}, index=[101, 102, 103, 201, 104])
X_toy, len_toy, ids_toy = build_uid_windows(toy.iloc[:0], toy, ['f'], window_size=3)
print(X_toy.shape, len_toy, ids_toy[:5])
print(X_toy[:, :, 0])


## PyTorch LSTM model

The model receives padded batches plus true sequence lengths. `pack_padded_sequence` prevents the LSTM from treating padding as real history.


In [ ]:
def local_roc_auc(y_true, y_score):
    if roc_auc_score is not None:
        return roc_auc_score(y_true, y_score)
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score).astype(float)
    order = np.argsort(s, kind='mergesort')
    ranks = np.empty(len(s), dtype=float)
    ranks[order] = np.arange(1, len(s) + 1)
    pos = y == 1
    n_pos = pos.sum()
    n_neg = len(y) - n_pos
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

class WindowDataset(Dataset):
    def __init__(self, X, lengths, y=None):
        self.X = torch.from_numpy(X.astype("float32"))
        self.lengths = torch.from_numpy(lengths.astype("int64"))
        self.y = None if y is None else torch.from_numpy(y.astype("float32"))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx], self.lengths[idx]
        return self.X[idx], self.lengths[idx], self.y[idx]

class FraudLSTM(nn.Module):
    def __init__(self, n_features, hidden=LSTM_UNITS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        last_hidden = h_n[-1]
        return self.head(last_hidden).squeeze(1)

def make_pos_weight(y):
    y = np.asarray(y).astype(int)
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    return torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=DEVICE)

def predict_proba(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            xb, lengths = batch[:2]
            xb = xb.to(DEVICE)
            lengths = lengths.to(DEVICE)
            logits = model(xb, lengths)
            preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(preds)


## Train with strict time-forward validation

For each validation month, preprocessing is fitted on prior months only. The validation sequence may use prior training rows and earlier validation row features as history, but never validation labels.


In [ ]:
oof = pd.Series(np.nan, index=X_train_base.index, dtype='float32')
fold_scores = []

for fold, (train_months, valid_month, idx_train, idx_valid) in enumerate(folds):
    print('\n' + '=' * 80)
    print(f'Fold {fold}: train months={train_months}, valid month={valid_month}')
    print(f'train rows={len(idx_train):,}, valid rows={len(idx_valid):,}')

    train_fold = X_train_base.loc[idx_train].copy()
    valid_fold = X_train_base.loc[idx_valid].copy()

    prep = fit_preprocessor(train_fold)
    feature_cols = prep['feature_cols']
    print(f'Using {len(feature_cols)} features')

    train_proc = apply_preprocessor(train_fold, prep)
    valid_proc = apply_preprocessor(valid_fold, prep)

    X_tr, len_tr, train_ids = build_uid_windows(train_proc.iloc[:0], train_proc, feature_cols, WINDOW_SIZE)
    X_va, len_va, valid_ids = build_uid_windows(train_proc, valid_proc, feature_cols, WINDOW_SIZE)

    y_tr = y_train.loc[train_ids].to_numpy(dtype='int8')
    y_va = y_train.loc[valid_ids].to_numpy(dtype='int8')

    print('X_tr:', X_tr.shape, 'X_va:', X_va.shape, 'fraud rate train/valid:', y_tr.mean(), y_va.mean())

    train_loader = DataLoader(
        WindowDataset(X_tr, len_tr, y_tr),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    valid_loader = DataLoader(
        WindowDataset(X_va, len_va, y_va),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = FraudLSTM(n_features=len(feature_cols)).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(y_tr))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

    best_auc = -np.inf
    best_pred = None
    patience_left = PATIENCE

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        n_seen = 0
        for xb, lengths, yb in train_loader:
            xb = xb.to(DEVICE)
            lengths = lengths.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb, lengths)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(yb)
            n_seen += len(yb)

        pred = predict_proba(model, valid_loader)
        auc = local_roc_auc(y_va, pred)
        print(f"epoch={epoch+1}, loss={total_loss/n_seen:.5f}, val_auc={auc:.6f}")

        if auc > best_auc:
            best_auc = auc
            best_pred = pred.astype("float32")
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("early stopping")
                break

    model.load_state_dict(best_state)
    oof.loc[valid_ids] = best_pred
    fold_scores.append({"fold": fold, "valid_month": valid_month, "auc": best_auc, "rows": len(valid_ids)})
    print(f"Fold {fold} AUC = {best_auc:.6f}")

    del train_fold, valid_fold, train_proc, valid_proc, X_tr, X_va, y_tr, y_va, model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

scored = oof.notna()
overall_auc = local_roc_auc(y_train.loc[oof.index[scored]], oof.loc[scored])
print('Strict PyTorch LSTM OOF AUC:', overall_auc)

oof_path = OUTPUT_DIR / f'oof_uid_pytorch_lstm_window{WINDOW_SIZE}_{"v" if INCLUDE_V else "no_v"}.csv'
pd.DataFrame({'TransactionID': oof.index, 'oof_uid_pytorch_lstm': oof.values}).to_csv(oof_path, index=False)
print('Wrote', oof_path)


## Optional: fit on all train and predict test

Set `RUN_TEST_PREDICTION = True` in the config cell to run this section. Test windows use train history and earlier test row features only; no test labels are used.


In [ ]:
if RUN_TEST_PREDICTION:
    prep = fit_preprocessor(X_train_base)
    feature_cols = prep['feature_cols']
    print(f'Final model using {len(feature_cols)} features')

    train_proc = apply_preprocessor(X_train_base, prep)
    test_proc = apply_preprocessor(X_test_base, prep)

    X_tr, len_tr, train_ids = build_uid_windows(train_proc.iloc[:0], train_proc, feature_cols, WINDOW_SIZE)
    X_te, len_te, test_ids = build_uid_windows(train_proc, test_proc, feature_cols, WINDOW_SIZE)
    y_tr = y_train.loc[train_ids].to_numpy(dtype='int8')

    train_loader = DataLoader(WindowDataset(X_tr, len_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(WindowDataset(X_te, len_te), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = FraudLSTM(n_features=len(feature_cols)).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(y_tr))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        n_seen = 0
        for xb, lengths, yb in train_loader:
            xb = xb.to(DEVICE)
            lengths = lengths.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb, lengths), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(yb)
            n_seen += len(yb)
        print(f"epoch={epoch+1}, loss={total_loss/n_seen:.5f}")

    test_pred = predict_proba(model, test_loader)
    sub = pd.DataFrame({'TransactionID': test_ids, 'isFraud': test_pred})
    sub = sub.sort_values('TransactionID')
    test_path = OUTPUT_DIR / f'test_uid_pytorch_lstm_window{WINDOW_SIZE}_{"v" if INCLUDE_V else "no_v"}.csv'
    sub.to_csv(test_path, index=False)
    print('Wrote', test_path)
else:
    print('RUN_TEST_PREDICTION is False; skipping final test prediction.')


## Notes for interpreting results

- This PyTorch notebook is intentionally equivalent in data framing to the Keras notebook.
- Padding is handled with sequence lengths and `pack_padded_sequence`.
- `uid_code` controls sequence grouping only; it is not a model feature.
- If the pickle files contain separately factorized train/test `uid` columns, this notebook ignores them and rebuilds `uid_raw_lstm` from `card1_addr1`, `day`, and `D1`.
- Start with `INCLUDE_V=False`; turn V columns on only after the full no-V pipeline runs.
